In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 1. LOAD DATA

In [52]:
with open("data.txt", "r") as f:
    lines = f.read().strip().split("\n")

# remove empty lines
lines = [line for line in lines if line.strip() != ""]

# 2. BUILD VOCAB

In [53]:
all_text = " ".join(lines)
words = all_text.split()

vocab = ["<UNK>"] + sorted(set(words))

word_to_index = {word: i for i, word in enumerate(vocab)}
index_to_word = {i: word for word, i in word_to_index.items()}

# 3. TOKENIZER

In [54]:
def encode(text):
    return [word_to_index.get(word, word_to_index["<UNK>"]) for word in text.split()]

def decode(indices):
    return " ".join([index_to_word.get(i, "<UNK>") for i in indices])


# 4. CREATE DATA

In [55]:
seq_length = 3

X, Y = [], []

for line in lines:
    encoded = encode(line)

    if len(encoded) <= seq_length:
        continue

    for i in range(len(encoded) - seq_length):
        X.append(encoded[i:i+seq_length])
        Y.append(encoded[i+1:i+seq_length+1])

X_tensor = torch.tensor(X)
Y_tensor = torch.tensor(Y)


# 5. MODEL

In [56]:
embed_dim = 16

embed = nn.Embedding(len(vocab), embed_dim)

Wq = nn.Linear(embed_dim, embed_dim)
Wk = nn.Linear(embed_dim, embed_dim)
Wv = nn.Linear(embed_dim, embed_dim)

fc = nn.Linear(embed_dim, len(vocab))

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    list(embed.parameters()) +
    list(Wq.parameters()) +
    list(Wk.parameters()) +
    list(Wv.parameters()) +
    list(fc.parameters()),
    lr=0.01
)


# 6. POSITIONAL ENCODING

In [57]:
def positional_encoding(seq_len, embed_dim):
    pe = torch.zeros(seq_len, embed_dim)

    for pos in range(seq_len):
        for i in range(embed_dim):
            pe[pos, i] = pos / (10000 ** (2 * (i//2) / embed_dim))

    return pe

# 7. TRAINING

In [58]:
for epoch in range(800):

    emb = embed(X_tensor)

    pe = positional_encoding(emb.shape[1], emb.shape[2])
    emb = emb + pe

    Q = Wq(emb)
    K = Wk(emb)
    V = Wv(emb)

    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(embed_dim)
    attention = F.softmax(scores, dim=-1)

    out = attention @ V
    output = fc(out)

    output = output.view(-1, len(vocab))
    Y_flat = Y_tensor.view(-1)

    loss = loss_fn(output, Y_flat)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 4.056193828582764
Epoch 50, Loss: 0.9507237076759338
Epoch 100, Loss: 0.44459807872772217
Epoch 150, Loss: 0.32373061776161194
Epoch 200, Loss: 0.28116869926452637
Epoch 250, Loss: 0.265880823135376
Epoch 300, Loss: 0.26140883564949036
Epoch 350, Loss: 0.2551819980144501
Epoch 400, Loss: 0.2238868772983551
Epoch 450, Loss: 0.21869789063930511
Epoch 500, Loss: 0.1958829164505005
Epoch 550, Loss: 0.17677433788776398
Epoch 600, Loss: 0.15792134404182434
Epoch 650, Loss: 0.1517316699028015
Epoch 700, Loss: 0.14655308425426483
Epoch 750, Loss: 0.1453460454940796


# 8. GENERATION FUNCTION

In [59]:
def generate(start_words, max_len=5):
    words = start_words.copy()

    for _ in range(max_len):

        # safe input (no mutation)
        input_words = words[-seq_length:]
        if len(input_words) < seq_length:
            input_words = ["<UNK>"] * (seq_length - len(input_words)) + input_words

        input_seq = encode(" ".join(input_words))
        input_tensor = torch.tensor([input_seq])

        emb = embed(input_tensor)
        pe = positional_encoding(emb.shape[1], emb.shape[2])
        emb = emb + pe

        Q = Wq(emb)
        K = Wk(emb)
        V = Wv(emb)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(embed_dim)
        attention = F.softmax(scores, dim=-1)

        out = attention @ V
        output = fc(out)

        last_output = output[0, -1]

        # repetition penalty (milder)
        for word in set(words):
            idx = word_to_index.get(word, None)
            if idx is not None:
                last_output[idx] /= 1.2

        probs = F.softmax(last_output / 0.8, dim=-1)
        predicted_index = torch.multinomial(probs, 1).item()

        next_word = index_to_word[predicted_index]

        if next_word == "<END>" and len(words) > seq_length:
            break

        if next_word == "<UNK>":
            break

        words.append(next_word)

    # return only generated answer
    return " ".join(words[len(start_words):])

# 9. TEST

In [62]:
while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    response = generate(user_input.split())
    print("Bot:", response)

KeyboardInterrupt: Interrupted by user